In [2]:
import os
import pandas as pd

# Input folder and file names
folder = r"C:\Android Mobile App\ICST2026_Ext\0-Data_Feb22"
input_base = "run_steps_v16_stage3_breakdown"
output_base = "run_steps_v16_stage3_breakdown_2"
target_full_name = "behnamparsa/toDoList"

# Build file paths (supports whether you included .csv or not)
input_path = os.path.join(folder, input_base)
if not input_path.lower().endswith(".csv"):
    input_path += ".csv"

output_path = os.path.join(folder, output_base)
if not output_path.lower().endswith(".csv"):
    output_path += ".csv"

# Read CSV
df = pd.read_csv(input_path)

# Find the correct column (expects a column named "full name", but also handles variations)
possible_cols = ["full name", "Full Name", "fullname", "full_name", "name"]
col = next((c for c in possible_cols if c in df.columns), None)

if col is None:
    raise ValueError(
        f"Could not find a 'full name' column. Available columns are: {list(df.columns)}"
    )

# Count and remove matching records
removed_count = (df[col].astype(str) == target_full_name).sum()
df_cleaned = df[df[col].astype(str) != target_full_name].copy()

# Save cleaned CSV
df_cleaned.to_csv(output_path, index=False)

# Report
print(f"Removed records: {removed_count}")
print(f"Saved cleaned file to: {output_path}")

C:\Users\gilla\AppData\Local\Temp\ipykernel_26880\2239805351.py:20: DtypeWarning: Columns (11,12,20,21,22,23,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


Removed records: 986
Saved cleaned file to: C:\Android Mobile App\ICST2026_Ext\0-Data_Feb22\run_steps_v16_stage3_breakdown_2.csv


In [ ]:
# API Telemetry 

In [1]:
# ============================================================
# Manual Review: Pull GitHub telemetry for one run and compare
# against detections stored in MainDataset.csv
#
# Target run:
# https://github.com/CatimaLoyalty/Android/actions/runs/20728761008
#
# Outputs:
#   Manual_Review_Run_20728761008/
#     github_run_summary.csv
#     github_jobs.csv
#     github_steps.csv
#     detected_invocation_steps.csv
#     main_dataset_row.csv
#     comparison_report.csv
#
# Optional:
#   Set GITHUB_TOKEN in your environment for higher rate limits:
#     PowerShell:
#       $env:GITHUB_TOKEN="ghp_..."
# ============================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import re
import time
import requests
import pandas as pd


# ------------------------------------------------------------
# User inputs
# ------------------------------------------------------------
RUN_URL = "https://github.com/CatimaLoyalty/Android/actions/runs/20728761008"

MAIN_DATASET_PATH = Path(r"C:\Android Mobile App\ICST2026_Ext\MainDataset.csv")

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_20728761008")
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def parse_github_run_url(run_url: str):
    """
    Extract owner, repo, and run_id from:
    https://github.com/{owner}/{repo}/actions/runs/{run_id}
    """
    parsed = urlparse(run_url)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2:4] != ["actions", "runs"]:
        raise ValueError(f"Not a recognized GitHub Actions run URL: {run_url}")

    owner = parts[0]
    repo = parts[1]
    run_id = parts[4]
    return owner, repo, run_id


def github_get(url: str, params=None):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "manual-review-telemetry-check",
    }

    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    resp = requests.get(url, headers=headers, params=params, timeout=60)

    # Simple rate-limit handling
    if resp.status_code == 403 and resp.headers.get("X-RateLimit-Remaining") == "0":
        reset_epoch = int(resp.headers.get("X-RateLimit-Reset", "0"))
        sleep_seconds = max(0, reset_epoch - int(time.time()) + 2)
        print(f"Rate limited. Sleeping for {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)
        resp = requests.get(url, headers=headers, params=params, timeout=60)

    resp.raise_for_status()
    return resp.json()


def github_paginated_get(url: str, item_key: str, params=None):
    """
    Fetch paginated GitHub REST results.
    For jobs endpoint, item_key='jobs'.
    """
    if params is None:
        params = {}

    params = dict(params)
    params["per_page"] = 100

    items = []
    page = 1

    while True:
        params["page"] = page
        payload = github_get(url, params=params)

        batch = payload.get(item_key, [])
        items.extend(batch)

        if len(batch) < params["per_page"]:
            break

        page += 1

    return items


def to_datetime_utc(series):
    return pd.to_datetime(series, errors="coerce", utc=True)


def seconds_between(start, end):
    if pd.isna(start) or pd.isna(end):
        return None
    return (end - start).total_seconds()


def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def contains_any(text, patterns):
    text = normalize_text(text).lower()
    return any(p.lower() in text for p in patterns)


# ------------------------------------------------------------
# Detection logic for this manual review
# ------------------------------------------------------------
INVOCATION_NAME_PATTERNS = [
    "run instrumented tests",
    "instrumented tests",
    "androidtest",
    "connected",
]

PROVIDER_PATTERNS = [
    "reactivecircus",
    "android-emulator-runner",
]


def is_invocation_step_from_telemetry(step_name: str):
    """
    GitHub jobs API exposes step names and timing, but not the step's `uses:`
    or `with:` content. Therefore, for API-only telemetry, detection is based
    primarily on step names.

    For this workflow, the invocation steps are expected to be:
      - Run instrumented tests (API 21)
      - Run instrumented tests (API 35)
    """
    name = normalize_text(step_name).lower()
    return contains_any(name, INVOCATION_NAME_PATTERNS)


def infer_step_activity_group(step_name: str):
    """
    Lightweight manual-review grouping, aligned with the workflow semantics.
    This is not meant to replace your main pipeline; it is only for checking
    whether the GitHub telemetry supports the stored detections.
    """
    name = normalize_text(step_name).lower()

    if "checkout" in name:
        return "Setup"

    if "wrapper-validation" in name or "openjdk" in name or "set up" in name:
        return "Setup"

    if "enable kvm" in name:
        return "Provision"

    if "build" == name or name.startswith("build"):
        return "Build"

    if "lint" in name:
        return "Test"

    if "unit tests" in name or "test" in name:
        return "Test"

    if "instrumented tests" in name or "androidtest" in name or "connected" in name:
        return "Invocation"

    if "archive" in name or "upload-artifact" in name:
        return "Post"

    return "Other"


# ------------------------------------------------------------
# Pull GitHub telemetry
# ------------------------------------------------------------
owner, repo, run_id = parse_github_run_url(RUN_URL)

api_base = "https://api.github.com"
run_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}"
jobs_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"

run = github_get(run_api_url)
jobs = github_paginated_get(jobs_api_url, item_key="jobs")

print(f"Pulled run: {run.get('html_url')}")
print(f"Jobs returned: {len(jobs)}")


# ------------------------------------------------------------
# Save run summary
# ------------------------------------------------------------
run_summary = {
    "full_name": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_id": run.get("workflow_id"),
    "name": run.get("name"),
    "path": run.get("path"),
    "html_url": run.get("html_url"),
    "run_number": run.get("run_number"),
    "run_attempt": run.get("run_attempt"),
    "event": run.get("event"),
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "head_branch": run.get("head_branch"),
    "head_sha": run.get("head_sha"),
    "created_at": run.get("created_at"),
    "run_started_at": run.get("run_started_at"),
    "updated_at": run.get("updated_at"),
}

run_summary_df = pd.DataFrame([run_summary])

run_summary_df["run_started_at_dt"] = to_datetime_utc(run_summary_df["run_started_at"])
run_summary_df["updated_at_dt"] = to_datetime_utc(run_summary_df["updated_at"])
run_summary_df["api_run_duration_seconds"] = run_summary_df.apply(
    lambda r: seconds_between(r["run_started_at_dt"], r["updated_at_dt"]),
    axis=1,
)

run_summary_df.to_csv(OUT_DIR / "github_run_summary.csv", index=False)


# ------------------------------------------------------------
# Flatten jobs and steps
# ------------------------------------------------------------
job_rows = []
step_rows = []

for job in jobs:
    job_started = pd.to_datetime(job.get("started_at"), errors="coerce", utc=True)
    job_completed = pd.to_datetime(job.get("completed_at"), errors="coerce", utc=True)

    job_rows.append({
        "job_id": job.get("id"),
        "job_run_id": job.get("run_id"),
        "job_name": job.get("name"),
        "job_status": job.get("status"),
        "job_conclusion": job.get("conclusion"),
        "job_started_at": job.get("started_at"),
        "job_completed_at": job.get("completed_at"),
        "job_duration_seconds": seconds_between(job_started, job_completed),
        "runner_name": job.get("runner_name"),
        "runner_group_name": job.get("runner_group_name"),
        "html_url": job.get("html_url"),
    })

    for step in job.get("steps", []):
        step_started = pd.to_datetime(step.get("started_at"), errors="coerce", utc=True)
        step_completed = pd.to_datetime(step.get("completed_at"), errors="coerce", utc=True)

        step_name = step.get("name")

        step_rows.append({
            "run_id": run.get("id"),
            "job_id": job.get("id"),
            "job_name": job.get("name"),
            "job_status": job.get("status"),
            "job_conclusion": job.get("conclusion"),
            "step_number": step.get("number"),
            "step_name": step_name,
            "step_status": step.get("status"),
            "step_conclusion": step.get("conclusion"),
            "step_started_at": step.get("started_at"),
            "step_completed_at": step.get("completed_at"),
            "step_duration_seconds": seconds_between(step_started, step_completed),
            "manual_activity_group": infer_step_activity_group(step_name),
            "manual_invocation_candidate_flag": is_invocation_step_from_telemetry(step_name),
        })

jobs_df = pd.DataFrame(job_rows)
steps_df = pd.DataFrame(step_rows)

jobs_df.to_csv(OUT_DIR / "github_jobs.csv", index=False)
steps_df.to_csv(OUT_DIR / "github_steps.csv", index=False)


# ------------------------------------------------------------
# Detect invocation steps from pulled telemetry
# ------------------------------------------------------------
detected_invocations = steps_df[steps_df["manual_invocation_candidate_flag"]].copy()

if not detected_invocations.empty:
    detected_invocations["step_started_at_dt"] = to_datetime_utc(detected_invocations["step_started_at"])
    detected_invocations["step_completed_at_dt"] = to_datetime_utc(detected_invocations["step_completed_at"])

    invocation_window_start = detected_invocations["step_started_at_dt"].min()
    invocation_window_end = detected_invocations["step_completed_at_dt"].max()
    invocation_window_seconds = seconds_between(invocation_window_start, invocation_window_end)

    detected_invocation_step_names = sorted(detected_invocations["step_name"].dropna().unique().tolist())
    detected_invocation_job_names = sorted(detected_invocations["job_name"].dropna().unique().tolist())
else:
    invocation_window_start = pd.NaT
    invocation_window_end = pd.NaT
    invocation_window_seconds = None
    detected_invocation_step_names = []
    detected_invocation_job_names = []

detected_invocations.to_csv(OUT_DIR / "detected_invocation_steps.csv", index=False)


# ------------------------------------------------------------
# Pull the corresponding row from MainDataset.csv
# ------------------------------------------------------------
main_df = pd.read_csv(MAIN_DATASET_PATH, low_memory=False)

# Prefer run_id match. Fall back to html_url if needed.
run_id_int = int(run_id)

if "run_id" in main_df.columns:
    dataset_match = main_df[main_df["run_id"].astype(str) == str(run_id_int)].copy()
else:
    dataset_match = pd.DataFrame()

if dataset_match.empty and "html_url" in main_df.columns:
    dataset_match = main_df[main_df["html_url"].astype(str).str.contains(str(run_id_int), regex=False, na=False)].copy()

if dataset_match.empty:
    print("WARNING: No matching row found in MainDataset.csv for this run.")
    dataset_row = pd.DataFrame()
else:
    dataset_row = dataset_match.head(1).copy()
    dataset_row.to_csv(OUT_DIR / "main_dataset_row.csv", index=False)


# ------------------------------------------------------------
# Build comparison report
# ------------------------------------------------------------
comparison_rows = []


def dataset_value(col):
    if dataset_row.empty or col not in dataset_row.columns:
        return None
    return dataset_row.iloc[0][col]


def add_comparison(field, github_value, dataset_col=None, dataset_val=None, note=""):
    if dataset_val is None and dataset_col is not None:
        dataset_val = dataset_value(dataset_col)

    comparison_rows.append({
        "check": field,
        "github_manual_review_value": github_value,
        "main_dataset_column": dataset_col,
        "main_dataset_value": dataset_val,
        "match_string_exact": (
            str(github_value) == str(dataset_val)
            if dataset_val is not None
            else None
        ),
        "note": note,
    })


# Basic run metadata checks
add_comparison("run id", run.get("id"), "run_id")
add_comparison("html url", run.get("html_url"), "html_url")
add_comparison("workflow id", run.get("workflow_id"), "workflow_id")
add_comparison("workflow path", run.get("path"), "workflow_path")
add_comparison("run number", run.get("run_number"), "run_number")
add_comparison("run attempt", run.get("run_attempt"), "run_attempt")
add_comparison("event", run.get("event"), "event")
add_comparison("status", run.get("status"), "status")
add_comparison("conclusion", run.get("conclusion"), "run_conclusion")
add_comparison("head branch", run.get("head_branch"), "head_branch")
add_comparison("head sha", run.get("head_sha"), "head_sha")

# Duration check
api_run_duration = run_summary_df.iloc[0]["api_run_duration_seconds"]
add_comparison(
    "API run duration seconds",
    api_run_duration,
    "study_run_duration_seconds",
    note="Computed as updated_at - run_started_at from the GitHub Actions run API."
)

# Job / step checks
add_comparison(
    "GitHub jobs returned",
    len(jobs_df),
    None,
    note="Number of expanded jobs returned by GitHub jobs API."
)

add_comparison(
    "Step telemetry available",
    len(steps_df) > 0,
    None,
    note="True means the GitHub jobs API returned step-level telemetry."
)

add_comparison(
    "Manual invocation candidate count",
    len(detected_invocations),
    "study_invocation_candidate_count_total",
    note="Detected from GitHub step names. The API does not expose `uses:` directly in job steps."
)

add_comparison(
    "Manual distinct invocation step names",
    " | ".join(detected_invocation_step_names),
    "study_invocation_candidate_step_names",
    note="Expected here: Run instrumented tests (API 21), Run instrumented tests (API 35)."
)

add_comparison(
    "Manual distinct invocation job names",
    " | ".join(detected_invocation_job_names),
    "study_invocation_candidate_job_names",
    note="Expected matrix-expanded build jobs, likely one for Foss and one for Gplay."
)

add_comparison(
    "Manual invocation window start",
    invocation_window_start.isoformat() if pd.notna(invocation_window_start) else None,
    "study_invocation_execution_window_started_at",
)

add_comparison(
    "Manual invocation window end",
    invocation_window_end.isoformat() if pd.notna(invocation_window_end) else None,
    "study_invocation_execution_window_ended_at",
)

add_comparison(
    "Manual invocation execution window seconds",
    invocation_window_seconds,
    "study_invocation_execution_window_direct_seconds",
    note="Computed as max completed_at minus min started_at across detected invocation steps."
)

# Style/provider expectations based on the provided workflow content
add_comparison(
    "Manual provider expectation",
    "ReactiveCircus/android-emulator-runner@v2",
    "third_party_provider_name",
    note="Based on the workflow YAML content supplied in the manual review request."
)

add_comparison(
    "Manual style expectation",
    "Android emulator / instrumented test workflow style",
    "style",
    note="The workflow has two ReactiveCircus android-emulator-runner invocations, API 21 and API 35, across a Foss/Gplay matrix."
)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUT_DIR / "comparison_report.csv", index=False)


# ------------------------------------------------------------
# Console summary
# ------------------------------------------------------------
print("\n=== Manual Review Summary ===")
print(f"Repository: {owner}/{repo}")
print(f"Run ID: {run_id}")
print(f"Workflow: {run.get('name')}")
print(f"Status/conclusion: {run.get('status')} / {run.get('conclusion')}")
print(f"Run attempt: {run.get('run_attempt')}")
print(f"Jobs returned: {len(jobs_df)}")
print(f"Steps returned: {len(steps_df)}")
print(f"Detected invocation steps: {len(detected_invocations)}")

print("\nDetected invocation step names:")
for x in detected_invocation_step_names:
    print(f"  - {x}")

print("\nDetected invocation job names:")
for x in detected_invocation_job_names:
    print(f"  - {x}")

print("\nInvocation window:")
print(f"  start:   {invocation_window_start}")
print(f"  end:     {invocation_window_end}")
print(f"  seconds: {invocation_window_seconds}")

if dataset_row.empty:
    print("\nNo row found in MainDataset.csv for this run.")
else:
    print("\nMainDataset row found and exported.")
    useful_cols = [
        "full_name",
        "run_id",
        "html_url",
        "style",
        "third_party_provider_name",
        "study_invocation_candidate_count_total",
        "study_invocation_candidate_step_names",
        "study_invocation_candidate_job_names",
        "study_matched_invocation_step_name",
        "study_matched_invocation_job_name",
        "study_invocation_execution_window_started_at",
        "study_invocation_execution_window_ended_at",
        "study_invocation_execution_window_direct_seconds",
    ]

    useful_cols = [c for c in useful_cols if c in dataset_row.columns]
    print(dataset_row[useful_cols].T)

print(f"\nFiles written to: {OUT_DIR}")

Pulled run: https://github.com/CatimaLoyalty/Android/actions/runs/20728761008
Jobs returned: 2

=== Manual Review Summary ===
Repository: CatimaLoyalty/Android
Run ID: 20728761008
Workflow: Android CI
Status/conclusion: completed / success
Run attempt: 1
Jobs returned: 2
Steps returned: 26
Detected invocation steps: 4

Detected invocation step names:
  - Run instrumented tests (API 21)
  - Run instrumented tests (API 35)

Detected invocation job names:
  - build (Foss)
  - build (Gplay)

Invocation window:
  start:   2026-01-05 20:58:34+00:00
  end:     2026-01-05 21:06:53+00:00
  seconds: 499.0

MainDataset row found and exported.
                                                                                               6746
full_name                                                                     CatimaLoyalty/Android
run_id                                                                                  20728761008
html_url                                          https://gi

In [2]:
# ============================================================
# Pull general GitHub Actions telemetry timeline for one run
#
# Target:
# https://github.com/CatimaLoyalty/Android/actions/runs/20728761008
#
# Output:
#   github_run_general_timeline_20728761008.csv
#
# Optional token:
#   PowerShell:
#     $env:GITHUB_TOKEN="ghp_..."
# ============================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import time
import requests
import pandas as pd


# ------------------------------------------------------------
# Input
# ------------------------------------------------------------
RUN_URL = "https://github.com/CatimaLoyalty/Android/actions/runs/20728761008"

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_20728761008")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "github_run_general_timeline_20728761008.csv"


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def parse_github_run_url(run_url: str):
    """
    Extract owner, repo, run_id from:
    https://github.com/{owner}/{repo}/actions/runs/{run_id}
    """
    parsed = urlparse(run_url)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2:4] != ["actions", "runs"]:
        raise ValueError(f"Invalid GitHub Actions run URL: {run_url}")

    owner = parts[0]
    repo = parts[1]
    run_id = parts[4]

    return owner, repo, run_id


def github_get(url: str, params=None):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "github-actions-run-telemetry-review",
    }

    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    response = requests.get(url, headers=headers, params=params, timeout=60)

    # Basic rate-limit handling
    if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
        reset_epoch = int(response.headers.get("X-RateLimit-Reset", "0"))
        sleep_seconds = max(0, reset_epoch - int(time.time()) + 2)
        print(f"Rate limited. Sleeping for {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)
        response = requests.get(url, headers=headers, params=params, timeout=60)

    response.raise_for_status()
    return response.json()


def github_paginated_get(url: str, item_key: str):
    """
    Fetch paginated GitHub REST API results.
    Example:
      jobs endpoint returns {"total_count": ..., "jobs": [...]}
    """
    all_items = []
    page = 1

    while True:
        payload = github_get(
            url,
            params={
                "per_page": 100,
                "page": page,
            },
        )

        batch = payload.get(item_key, [])
        all_items.extend(batch)

        if len(batch) < 100:
            break

        page += 1

    return all_items


def dt(x):
    return pd.to_datetime(x, errors="coerce", utc=True)


def seconds_between(start, end):
    start = dt(start)
    end = dt(end)

    if pd.isna(start) or pd.isna(end):
        return None

    return (end - start).total_seconds()


def seconds_from_run_start(timestamp, run_start):
    timestamp = dt(timestamp)
    run_start = dt(run_start)

    if pd.isna(timestamp) or pd.isna(run_start):
        return None

    return (timestamp - run_start).total_seconds()


# ------------------------------------------------------------
# Pull GitHub API telemetry
# ------------------------------------------------------------
owner, repo, run_id = parse_github_run_url(RUN_URL)

api_base = "https://api.github.com"
run_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}"
jobs_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"

run = github_get(run_api_url)
jobs = github_paginated_get(jobs_api_url, item_key="jobs")

run_start = run.get("run_started_at") or run.get("created_at")
run_end = run.get("updated_at")

print(f"Repository: {owner}/{repo}")
print(f"Run ID: {run_id}")
print(f"Workflow: {run.get('name')}")
print(f"Run status/conclusion: {run.get('status')} / {run.get('conclusion')}")
print(f"Run start: {run_start}")
print(f"Run end:   {run_end}")
print(f"Jobs pulled: {len(jobs)}")


# ------------------------------------------------------------
# Build one general timeline table
# ------------------------------------------------------------
timeline_rows = []

# Run start row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN START",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_start,
    "completed_at": run_end,
    "duration_seconds": seconds_between(run_start, run_end),
    "seconds_from_run_start": 0,
    "html_url": run.get("html_url"),
})

# Job and step rows
for job in jobs:
    job_id = job.get("id")
    job_name = job.get("name")
    job_start = job.get("started_at")
    job_end = job.get("completed_at")

    # Job row
    timeline_rows.append({
        "event_type": "job",
        "event_name": f"JOB: {job_name}",
        "repo": f"{owner}/{repo}",
        "run_id": run.get("id"),
        "workflow_name": run.get("name"),
        "workflow_path": run.get("path"),
        "run_attempt": run.get("run_attempt"),
        "run_event": run.get("event"),
        "branch": run.get("head_branch"),
        "sha": run.get("head_sha"),
        "job_id": job_id,
        "job_name": job_name,
        "step_number": None,
        "step_name": None,
        "status": job.get("status"),
        "conclusion": job.get("conclusion"),
        "started_at": job_start,
        "completed_at": job_end,
        "duration_seconds": seconds_between(job_start, job_end),
        "seconds_from_run_start": seconds_from_run_start(job_start, run_start),
        "html_url": job.get("html_url"),
    })

    # Step rows
    for step in job.get("steps", []):
        step_name = step.get("name")
        step_start = step.get("started_at")
        step_end = step.get("completed_at")

        timeline_rows.append({
            "event_type": "step",
            "event_name": f"STEP: {job_name} / {step_name}",
            "repo": f"{owner}/{repo}",
            "run_id": run.get("id"),
            "workflow_name": run.get("name"),
            "workflow_path": run.get("path"),
            "run_attempt": run.get("run_attempt"),
            "run_event": run.get("event"),
            "branch": run.get("head_branch"),
            "sha": run.get("head_sha"),
            "job_id": job_id,
            "job_name": job_name,
            "step_number": step.get("number"),
            "step_name": step_name,
            "status": step.get("status"),
            "conclusion": step.get("conclusion"),
            "started_at": step_start,
            "completed_at": step_end,
            "duration_seconds": seconds_between(step_start, step_end),
            "seconds_from_run_start": seconds_from_run_start(step_start, run_start),
            "html_url": job.get("html_url"),
        })

# Run end row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN END",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_end,
    "completed_at": run_end,
    "duration_seconds": 0,
    "seconds_from_run_start": seconds_from_run_start(run_end, run_start),
    "html_url": run.get("html_url"),
})


timeline_df = pd.DataFrame(timeline_rows)

# Convert timestamps for sorting
timeline_df["_sort_started_at"] = pd.to_datetime(
    timeline_df["started_at"],
    errors="coerce",
    utc=True,
)

timeline_df["_event_order"] = timeline_df["event_type"].map({
    "run": 0,
    "job": 1,
    "step": 2,
}).fillna(9)

timeline_df = timeline_df.sort_values(
    by=["_sort_started_at", "_event_order", "job_name", "step_number"],
    na_position="last",
).drop(columns=["_sort_started_at", "_event_order"])

timeline_df.to_csv(OUT_FILE, index=False)

print(f"\nSaved general timeline to:")
print(OUT_FILE)


# ------------------------------------------------------------
# Print a readable console view
# ------------------------------------------------------------
print("\n=== General Run Timeline ===")

display_cols = [
    "event_type",
    "job_name",
    "step_number",
    "step_name",
    "status",
    "conclusion",
    "started_at",
    "completed_at",
    "duration_seconds",
    "seconds_from_run_start",
]

print(
    timeline_df[display_cols]
    .to_string(index=False, max_colwidth=60)
)

Repository: CatimaLoyalty/Android
Run ID: 20728761008
Workflow: Android CI
Run status/conclusion: completed / success
Run start: 2026-01-05T20:50:38Z
Run end:   2026-01-05T21:06:57Z
Jobs pulled: 2

Saved general timeline to:
C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_20728761008\github_run_general_timeline_20728761008.csv

=== General Run Timeline ===
event_type      job_name  step_number                                step_name    status conclusion           started_at         completed_at  duration_seconds  seconds_from_run_start
       run          None          NaN                                     None completed    success 2026-01-05T20:50:38Z 2026-01-05T21:06:57Z             979.0                     0.0
       job  build (Foss)          NaN                                     None completed    success 2026-01-05T20:50:41Z 2026-01-05T21:06:56Z             975.0                     3.0
       job build (Gplay)      

In [3]:
# ============================================================
# Pull general GitHub Actions telemetry timeline for one run
#
# Target:
# https://github.com/CatimaLoyalty/Android/actions/runs/20728761008
#
# Output:
#   github_run_general_timeline_20728761008.csv
#
# Optional token:
#   PowerShell:
#     $env:GITHUB_TOKEN="ghp_..."
# ============================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import time
import requests
import pandas as pd


# ------------------------------------------------------------
# Input
# ------------------------------------------------------------
RUN_URL = "https://github.com/EventFahrplan/EventFahrplan/actions/runs/20495275523"

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_20495275523")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "github_run_general_timeline_20495275523.csv"


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def parse_github_run_url(run_url: str):
    """
    Extract owner, repo, run_id from:
    https://github.com/{owner}/{repo}/actions/runs/{run_id}
    """
    parsed = urlparse(run_url)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2:4] != ["actions", "runs"]:
        raise ValueError(f"Invalid GitHub Actions run URL: {run_url}")

    owner = parts[0]
    repo = parts[1]
    run_id = parts[4]

    return owner, repo, run_id


def github_get(url: str, params=None):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "github-actions-run-telemetry-review",
    }

    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    response = requests.get(url, headers=headers, params=params, timeout=60)

    # Basic rate-limit handling
    if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
        reset_epoch = int(response.headers.get("X-RateLimit-Reset", "0"))
        sleep_seconds = max(0, reset_epoch - int(time.time()) + 2)
        print(f"Rate limited. Sleeping for {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)
        response = requests.get(url, headers=headers, params=params, timeout=60)

    response.raise_for_status()
    return response.json()


def github_paginated_get(url: str, item_key: str):
    """
    Fetch paginated GitHub REST API results.
    Example:
      jobs endpoint returns {"total_count": ..., "jobs": [...]}
    """
    all_items = []
    page = 1

    while True:
        payload = github_get(
            url,
            params={
                "per_page": 100,
                "page": page,
            },
        )

        batch = payload.get(item_key, [])
        all_items.extend(batch)

        if len(batch) < 100:
            break

        page += 1

    return all_items


def dt(x):
    return pd.to_datetime(x, errors="coerce", utc=True)


def seconds_between(start, end):
    start = dt(start)
    end = dt(end)

    if pd.isna(start) or pd.isna(end):
        return None

    return (end - start).total_seconds()


def seconds_from_run_start(timestamp, run_start):
    timestamp = dt(timestamp)
    run_start = dt(run_start)

    if pd.isna(timestamp) or pd.isna(run_start):
        return None

    return (timestamp - run_start).total_seconds()


# ------------------------------------------------------------
# Pull GitHub API telemetry
# ------------------------------------------------------------
owner, repo, run_id = parse_github_run_url(RUN_URL)

api_base = "https://api.github.com"
run_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}"
jobs_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"

run = github_get(run_api_url)
jobs = github_paginated_get(jobs_api_url, item_key="jobs")

run_start = run.get("run_started_at") or run.get("created_at")
run_end = run.get("updated_at")

print(f"Repository: {owner}/{repo}")
print(f"Run ID: {run_id}")
print(f"Workflow: {run.get('name')}")
print(f"Run status/conclusion: {run.get('status')} / {run.get('conclusion')}")
print(f"Run start: {run_start}")
print(f"Run end:   {run_end}")
print(f"Jobs pulled: {len(jobs)}")


# ------------------------------------------------------------
# Build one general timeline table
# ------------------------------------------------------------
timeline_rows = []

# Run start row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN START",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_start,
    "completed_at": run_end,
    "duration_seconds": seconds_between(run_start, run_end),
    "seconds_from_run_start": 0,
    "html_url": run.get("html_url"),
})

# Job and step rows
for job in jobs:
    job_id = job.get("id")
    job_name = job.get("name")
    job_start = job.get("started_at")
    job_end = job.get("completed_at")

    # Job row
    timeline_rows.append({
        "event_type": "job",
        "event_name": f"JOB: {job_name}",
        "repo": f"{owner}/{repo}",
        "run_id": run.get("id"),
        "workflow_name": run.get("name"),
        "workflow_path": run.get("path"),
        "run_attempt": run.get("run_attempt"),
        "run_event": run.get("event"),
        "branch": run.get("head_branch"),
        "sha": run.get("head_sha"),
        "job_id": job_id,
        "job_name": job_name,
        "step_number": None,
        "step_name": None,
        "status": job.get("status"),
        "conclusion": job.get("conclusion"),
        "started_at": job_start,
        "completed_at": job_end,
        "duration_seconds": seconds_between(job_start, job_end),
        "seconds_from_run_start": seconds_from_run_start(job_start, run_start),
        "html_url": job.get("html_url"),
    })

    # Step rows
    for step in job.get("steps", []):
        step_name = step.get("name")
        step_start = step.get("started_at")
        step_end = step.get("completed_at")

        timeline_rows.append({
            "event_type": "step",
            "event_name": f"STEP: {job_name} / {step_name}",
            "repo": f"{owner}/{repo}",
            "run_id": run.get("id"),
            "workflow_name": run.get("name"),
            "workflow_path": run.get("path"),
            "run_attempt": run.get("run_attempt"),
            "run_event": run.get("event"),
            "branch": run.get("head_branch"),
            "sha": run.get("head_sha"),
            "job_id": job_id,
            "job_name": job_name,
            "step_number": step.get("number"),
            "step_name": step_name,
            "status": step.get("status"),
            "conclusion": step.get("conclusion"),
            "started_at": step_start,
            "completed_at": step_end,
            "duration_seconds": seconds_between(step_start, step_end),
            "seconds_from_run_start": seconds_from_run_start(step_start, run_start),
            "html_url": job.get("html_url"),
        })

# Run end row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN END",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_end,
    "completed_at": run_end,
    "duration_seconds": 0,
    "seconds_from_run_start": seconds_from_run_start(run_end, run_start),
    "html_url": run.get("html_url"),
})


timeline_df = pd.DataFrame(timeline_rows)

# Convert timestamps for sorting
timeline_df["_sort_started_at"] = pd.to_datetime(
    timeline_df["started_at"],
    errors="coerce",
    utc=True,
)

timeline_df["_event_order"] = timeline_df["event_type"].map({
    "run": 0,
    "job": 1,
    "step": 2,
}).fillna(9)

timeline_df = timeline_df.sort_values(
    by=["_sort_started_at", "_event_order", "job_name", "step_number"],
    na_position="last",
).drop(columns=["_sort_started_at", "_event_order"])

timeline_df.to_csv(OUT_FILE, index=False)

print(f"\nSaved general timeline to:")
print(OUT_FILE)


# ------------------------------------------------------------
# Print a readable console view
# ------------------------------------------------------------
print("\n=== General Run Timeline ===")

display_cols = [
    "event_type",
    "job_name",
    "step_number",
    "step_name",
    "status",
    "conclusion",
    "started_at",
    "completed_at",
    "duration_seconds",
    "seconds_from_run_start",
]

print(
    timeline_df[display_cols]
    .to_string(index=False, max_colwidth=60)
)

Repository: EventFahrplan/EventFahrplan
Run ID: 20495275523
Workflow: Build
Run status/conclusion: completed / success
Run start: 2025-12-24T23:07:23Z
Run end:   2025-12-24T23:17:46Z
Jobs pulled: 2

Saved general timeline to:
C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_20495275523\github_run_general_timeline_20495275523.csv

=== General Run Timeline ===
event_type           job_name  step_number                                                    step_name    status conclusion           started_at         completed_at  duration_seconds  seconds_from_run_start
       run               None          NaN                                                         None completed    success 2025-12-24T23:07:23Z 2025-12-24T23:17:46Z             623.0                     0.0
       job              build          NaN                                                         None completed    success 2025-12-24T23:07:26Z 2025-12-24T23:17:

In [ ]:
#GMD sample: https://github.com/FooIbar/EhViewer/actions/runs/22557070276/job/65336353665


In [ ]:
# ============================================================
# Pull general GitHub Actions telemetry timeline for one run
#
# Target:
# https://github.com/FooIbar/EhViewer/actions/runs/22557070276/job/65336353665
#
# Output:
#   github_run_general_timeline_20728761008.csv
#
# Optional token:
#   PowerShell:
#     $env:GITHUB_TOKEN="ghp_..."
# ============================================================

from pathlib import Path
from urllib.parse import urlparse
import os
import time
import requests
import pandas as pd


# ------------------------------------------------------------
# Input
# ------------------------------------------------------------
RUN_URL = "https://github.com/FooIbar/EhViewer/actions/runs/22557070276/job/65336353665"

OUT_DIR = Path(r"C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_65336353665")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "github_run_general_timeline_65336353665.csv"


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def parse_github_run_url(run_url: str):
    """
    Extract owner, repo, run_id from:
    https://github.com/{owner}/{repo}/actions/runs/{run_id}
    """
    parsed = urlparse(run_url)
    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2:4] != ["actions", "runs"]:
        raise ValueError(f"Invalid GitHub Actions run URL: {run_url}")

    owner = parts[0]
    repo = parts[1]
    run_id = parts[4]

    return owner, repo, run_id


def github_get(url: str, params=None):
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "github-actions-run-telemetry-review",
    }

    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    response = requests.get(url, headers=headers, params=params, timeout=60)

    # Basic rate-limit handling
    if response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
        reset_epoch = int(response.headers.get("X-RateLimit-Reset", "0"))
        sleep_seconds = max(0, reset_epoch - int(time.time()) + 2)
        print(f"Rate limited. Sleeping for {sleep_seconds} seconds...")
        time.sleep(sleep_seconds)
        response = requests.get(url, headers=headers, params=params, timeout=60)

    response.raise_for_status()
    return response.json()


def github_paginated_get(url: str, item_key: str):
    """
    Fetch paginated GitHub REST API results.
    Example:
      jobs endpoint returns {"total_count": ..., "jobs": [...]}
    """
    all_items = []
    page = 1

    while True:
        payload = github_get(
            url,
            params={
                "per_page": 100,
                "page": page,
            },
        )

        batch = payload.get(item_key, [])
        all_items.extend(batch)

        if len(batch) < 100:
            break

        page += 1

    return all_items


def dt(x):
    return pd.to_datetime(x, errors="coerce", utc=True)


def seconds_between(start, end):
    start = dt(start)
    end = dt(end)

    if pd.isna(start) or pd.isna(end):
        return None

    return (end - start).total_seconds()


def seconds_from_run_start(timestamp, run_start):
    timestamp = dt(timestamp)
    run_start = dt(run_start)

    if pd.isna(timestamp) or pd.isna(run_start):
        return None

    return (timestamp - run_start).total_seconds()


# ------------------------------------------------------------
# Pull GitHub API telemetry
# ------------------------------------------------------------
owner, repo, run_id = parse_github_run_url(RUN_URL)

api_base = "https://api.github.com"
run_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}"
jobs_api_url = f"{api_base}/repos/{owner}/{repo}/actions/runs/{run_id}/jobs"

run = github_get(run_api_url)
jobs = github_paginated_get(jobs_api_url, item_key="jobs")

run_start = run.get("run_started_at") or run.get("created_at")
run_end = run.get("updated_at")

print(f"Repository: {owner}/{repo}")
print(f"Run ID: {run_id}")
print(f"Workflow: {run.get('name')}")
print(f"Run status/conclusion: {run.get('status')} / {run.get('conclusion')}")
print(f"Run start: {run_start}")
print(f"Run end:   {run_end}")
print(f"Jobs pulled: {len(jobs)}")


# ------------------------------------------------------------
# Build one general timeline table
# ------------------------------------------------------------
timeline_rows = []

# Run start row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN START",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_start,
    "completed_at": run_end,
    "duration_seconds": seconds_between(run_start, run_end),
    "seconds_from_run_start": 0,
    "html_url": run.get("html_url"),
})

# Job and step rows
for job in jobs:
    job_id = job.get("id")
    job_name = job.get("name")
    job_start = job.get("started_at")
    job_end = job.get("completed_at")

    # Job row
    timeline_rows.append({
        "event_type": "job",
        "event_name": f"JOB: {job_name}",
        "repo": f"{owner}/{repo}",
        "run_id": run.get("id"),
        "workflow_name": run.get("name"),
        "workflow_path": run.get("path"),
        "run_attempt": run.get("run_attempt"),
        "run_event": run.get("event"),
        "branch": run.get("head_branch"),
        "sha": run.get("head_sha"),
        "job_id": job_id,
        "job_name": job_name,
        "step_number": None,
        "step_name": None,
        "status": job.get("status"),
        "conclusion": job.get("conclusion"),
        "started_at": job_start,
        "completed_at": job_end,
        "duration_seconds": seconds_between(job_start, job_end),
        "seconds_from_run_start": seconds_from_run_start(job_start, run_start),
        "html_url": job.get("html_url"),
    })

    # Step rows
    for step in job.get("steps", []):
        step_name = step.get("name")
        step_start = step.get("started_at")
        step_end = step.get("completed_at")

        timeline_rows.append({
            "event_type": "step",
            "event_name": f"STEP: {job_name} / {step_name}",
            "repo": f"{owner}/{repo}",
            "run_id": run.get("id"),
            "workflow_name": run.get("name"),
            "workflow_path": run.get("path"),
            "run_attempt": run.get("run_attempt"),
            "run_event": run.get("event"),
            "branch": run.get("head_branch"),
            "sha": run.get("head_sha"),
            "job_id": job_id,
            "job_name": job_name,
            "step_number": step.get("number"),
            "step_name": step_name,
            "status": step.get("status"),
            "conclusion": step.get("conclusion"),
            "started_at": step_start,
            "completed_at": step_end,
            "duration_seconds": seconds_between(step_start, step_end),
            "seconds_from_run_start": seconds_from_run_start(step_start, run_start),
            "html_url": job.get("html_url"),
        })

# Run end row
timeline_rows.append({
    "event_type": "run",
    "event_name": "RUN END",
    "repo": f"{owner}/{repo}",
    "run_id": run.get("id"),
    "workflow_name": run.get("name"),
    "workflow_path": run.get("path"),
    "run_attempt": run.get("run_attempt"),
    "run_event": run.get("event"),
    "branch": run.get("head_branch"),
    "sha": run.get("head_sha"),
    "job_id": None,
    "job_name": None,
    "step_number": None,
    "step_name": None,
    "status": run.get("status"),
    "conclusion": run.get("conclusion"),
    "started_at": run_end,
    "completed_at": run_end,
    "duration_seconds": 0,
    "seconds_from_run_start": seconds_from_run_start(run_end, run_start),
    "html_url": run.get("html_url"),
})


timeline_df = pd.DataFrame(timeline_rows)

# Convert timestamps for sorting
timeline_df["_sort_started_at"] = pd.to_datetime(
    timeline_df["started_at"],
    errors="coerce",
    utc=True,
)

timeline_df["_event_order"] = timeline_df["event_type"].map({
    "run": 0,
    "job": 1,
    "step": 2,
}).fillna(9)

timeline_df = timeline_df.sort_values(
    by=["_sort_started_at", "_event_order", "job_name", "step_number"],
    na_position="last",
).drop(columns=["_sort_started_at", "_event_order"])

timeline_df.to_csv(OUT_FILE, index=False)

print(f"\nSaved general timeline to:")
print(OUT_FILE)


# ------------------------------------------------------------
# Print a readable console view
# ------------------------------------------------------------
print("\n=== General Run Timeline ===")

display_cols = [
    "event_type",
    "job_name",
    "step_number",
    "step_name",
    "status",
    "conclusion",
    "started_at",
    "completed_at",
    "duration_seconds",
    "seconds_from_run_start",
]

print(
    timeline_df[display_cols]
    .to_string(index=False, max_colwidth=60)
)



Repository: FooIbar/EhViewer
Run ID: 22557070276
Workflow: Baseline profile generation
Run status/conclusion: completed / success
Run start: 2026-03-02T00:55:01Z
Run end:   2026-03-02T01:14:54Z
Jobs pulled: 1

Saved general timeline to:
C:\Android Mobile App\ICST2026_Ext\0.2-Validation\Step-2-Stratified_Sample\Manual_Checks\Manual_Review_Run_65336353665\github_run_general_timeline_65336353665.csv

=== General Run Timeline ===
event_type         job_name  step_number                 step_name    status conclusion           started_at         completed_at  duration_seconds  seconds_from_run_start
       run             None          NaN                      None completed    success 2026-03-02T00:55:01Z 2026-03-02T01:14:54Z            1193.0                     0.0
       job baseline-profile          NaN                      None completed    success 2026-03-02T00:55:03Z 2026-03-02T01:14:53Z            1190.0                     2.0
      step baseline-profile          1.0              